In [2]:
import pandas as pd

disruptions_df = pd.read_csv('final_feature_df.csv')

In [3]:
disruptions_df

,Date,source,destination,source_degree,source_weighted_degree,source_avg_distance,target_degree,target_weighted_degree,target_avg_distance,common_neighbors,...,T10N,FHVEC,Disrupted,Days_since_last_disruption,Num_prev_disruptions,Ratio,Total_disruptions,Total_rides,Previous_causes_vec,Most_recent_causes_vec
0,2019-01-01,'s-Hertogenbosch,Roosendaal,14,83,31.714286,13,52,39.230769,2,...,28.0,40.0,True,0,0,0.000000,1,2,[0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. ...,[0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. ...
1,2019-01-01,'s-Hertogenbosch,Utrecht Centraal,14,83,31.714286,36,243,21.305556,2,...,28.0,40.0,True,0,0,0.000000,2,1,[0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. ...,[0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 1. 0. ...
2,2019-01-01,Den Haag Centraal,Eindhoven Centraal,29,253,52.793103,21,116,40.142857,3,...,50.0,52.0,True,0,0,0.000000,1,32,[0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. ...,[0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. ...
3,2019-01-01,Den Haag Centraal,Utrecht Centraal,29,253,52.793103,36,243,21.305556,10,...,50.0,52.0,True,0,0,0.000000,1,21,[0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. ...,[0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. ...
4,2019-01-01,Eindhoven Centraal,Den Haag Centraal,21,116,40.142857,29,253,52.793103,3,...,28.0,40.0,True,0,0,0.000000,1,32,[0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. ...,[0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. ...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
575148,2024-12-31,Zwolle,Lelystad Centrum,23,104,53.608696,15,78,59.133333,4,...,29.0,82.0,False,16,41,0.807882,41,5075,[0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. ...,[0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. ...
575149,2024-12-31,Zwolle,Nijmegen,23,104,53.608696,19,64,56.000000,7,...,29.0,82.0,False,34,29,0.975774,29,2972,[0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. ...,[0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. ...
575150,2024-12-31,Zwolle,Roosendaal,23,104,53.608696,18,83,53.277778,4,...,29.0,82.0,False,9,380,0.595229,380,63841,[0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. ...,[0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. ...
575151,2024-12-31,Zwolle,Utrecht Centraal,23,104,53.608696,40,226,15.375000,5,...,29.0,82.0,False,12,308,0.422827,308,72843,[0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. ...,[0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. ...


In [18]:
import networkx as nx

# Ensure the 'Date' column is in datetime format
disruptions_df['Date'] = pd.to_datetime(disruptions_df['Date'])

# Get unique dates and stations
unique_dates = disruptions_df['Date'].dt.date.unique()
sources = disruptions_df['source'].unique()
destinations = disruptions_df['destination'].unique()
stations = list(set(sources).union(set(destinations)))

# Function to create daily graph and calculate node features
def create_daily_graph(date, df):
    # Filter data for the specific date
    daily_df = df[df['Date'].dt.date == date]
    
    # Create graph
    G = nx.DiGraph()
    G.add_nodes_from(stations)
    
    # Add weighted edges
    for _, row in daily_df.iterrows():
        G.add_edge(
            row['source'], 
            row['destination'], 
            weight=row['weight'], 
            distance=row['distance']
        )
        # print(f"Added edge from {row['source']} to {row['target']} with weight {row['Rides planned']} and distance {row['distance']}")
    
    # Calculate node features
    node_features = {}
    for node in G.nodes():
        # Degree (number of connections)
        degree = G.degree(node)
        
        # Weighted degree (sum of rides planned)
        weighted_degree = sum(
            G[node][neighbor]['weight'] 
            for neighbor in G.neighbors(node)
        )
        
        # Average distance to neighbors
        avg_distance = (
            sum(G[node][neighbor]['distance'] for neighbor in G.neighbors(node)) / degree
            if degree > 0 else 0
        )
        
        node_features[node] = {
            'degree': degree,
            'weighted_degree': weighted_degree,
            'avg_distance': avg_distance
        }
    # Calculate edge features
    edge_features = {}
    for edge in G.edges():
        source, target = edge
        row = daily_df[(daily_df['source'] == source) & (daily_df['destination'] == target)]
        if not row.empty:
            edge_features[edge] = row.iloc[0].drop(['Date', 'source', 'destination', 'Disrupted', 'Total_disruptions', 'Total_rides', 'Previous_causes_vec', 'Most_recent_causes_vec']).to_dict()
        else:
            edge_features[edge] = {}  # Handle cases where no matching row is found
    
    return G, node_features, edge_features

example_date = unique_dates[0]
G, node_features, edge_features = create_daily_graph(example_date, disruptions_df)
daily_df = disruptions_df[disruptions_df['Date'].dt.date == example_date]

In [19]:
print(edge_features)

{('Eindhoven Centraal', 'Den Haag Centraal'): {'source_degree': 21, 'source_weighted_degree': 116, 'source_avg_distance': 40.142857142857146, 'target_degree': 29, 'target_weighted_degree': 253, 'target_avg_distance': 52.793103448275865, 'common_neighbors': 3, 'jaccard_coefficient': 0.1111111111111111, 'preferential_attachment': 609, 'adamic_adar_index': 1.3050640741175197, 'resource_allocation_index': 0.3019230769230769, 'weight': 32, 'distance': 131.0, 'RH': 12.0, 'SQ': 7.0, 'TG': 70.0, 'TN': 42.0, 'TX': 87.0, 'RHX': 4.0, 'VVX': 74.0, 'T10N': 28.0, 'FHVEC': 40.0, 'Days_since_last_disruption': 0, 'Num_prev_disruptions': 0, 'Ratio': 0.0}, ('Rotterdam Centraal', 'Amsterdam Centraal'): {'source_degree': 21, 'source_weighted_degree': 134, 'source_avg_distance': 53.0, 'target_degree': 49, 'target_weighted_degree': 437, 'target_avg_distance': 41.30612244897959, 'common_neighbors': 8, 'jaccard_coefficient': 0.25, 'preferential_attachment': 1029, 'adamic_adar_index': 5.091285633253196, 'resour

In [20]:
# Dictionary to store daily graphs and features
daily_graphs = {}
daily_features = {}
daily_edge_features = {}

# Process each day
for date in unique_dates:
    G, features, edge_features = create_daily_graph(date, disruptions_df)
    daily_graphs[date] = G
    daily_features[date] = features
    daily_edge_features[date] = edge_features

In [22]:
trajectories_dict = {}
for date in unique_dates:
    daily_df = disruptions_df[disruptions_df['Date'].dt.date == date]
    G = daily_graphs[date]
    trajectories_dict[date] = []  # Initialize an empty list for each date
    for edge in G.edges():
        source, target = edge
        disrupted = daily_df[(daily_df['source'] == source) & (daily_df['destination'] == target)]['Disrupted'].values
        disrupted = disrupted[0] if len(disrupted) > 0 else False  # Handle cases where no match is found
        trajectories_dict[date].append((edge, disrupted))

In [69]:
from torch_geometric.data import Data
import torch

# Function to convert NetworkX graph and features to Data object
def graph_to_data(date, G, node_features, edge_features, trajectories):
    # Node features
    nodes = list(G.nodes())
    # Extract feature values from the nested dictionaries
    x = torch.tensor([list(node_features[node].values()) for node in nodes], dtype=torch.float)
    
    # Edge index and edge features
    edges = list(G.edges())
    edge_index = torch.tensor([[nodes.index(u), nodes.index(v)] for u, v in edges], dtype=torch.long).t()
    edge_attr = torch.tensor([list(edge_features.get((u, v)).values()) 
                              if edge_features.get((u, v)) else [0]*len(next(iter(edge_features.values())).values())
                              for u, v in edges], dtype=torch.float)
    
    # Create list of Data objects, one per trajectory
    data_list = []
    for traj, label in trajectories:
        # Convert trajectory to node indices
        traj_indices = torch.tensor([nodes.index(node) for node in traj], dtype=torch.long)
        data = Data(x=x, edge_index=edge_index, edge_attr=edge_attr, 
                    trajectory=traj_indices, y=torch.tensor([label], dtype=torch.float))
        data_list.append(data)
    
    return data_list

# Create dataset
dataset = []
for date in unique_dates:
    G = daily_graphs[date]
    node_features = daily_features[date]  # Assumes dict {node: feature_vector}
    edge_features = daily_edge_features[date]  # Assumes dict {(u, v): feature_vector}
    trajectories = trajectories_dict[date]
    daily_data = graph_to_data(date, G, node_features, edge_features, trajectories)
    dataset.extend(daily_data)

print(f"Total dataset size: {len(dataset)}")
print(f"Sample data: x={dataset[1].x.shape}, edge_index={dataset[1].edge_index.shape}, "
      f"trajectory={dataset[1].trajectory}, y={dataset[1].y}")

Total dataset size: 575153
Sample data: x=torch.Size([244, 3]), edge_index=torch.Size([2, 6]), trajectory=tensor([83, 73]), y=tensor([1.])


In [ ]:
import torch.nn as nn
from torch_geometric.nn import GATConv
import torch.nn.functional as F
from torch_geometric.nn import GCNConv  # Example GNN layer
from torch_geometric.data import Data

class GNN_LSTM(nn.Module):
    def __init__(self, node_feature_dim, edge_feature_dim, hidden_dim, lstm_hidden_dim, output_dim):
        super(GNN_LSTM, self).__init__()
        self.conv1 = GATConv(node_feature_dim, hidden_dim, edge_dim=edge_feature_dim)
        self.conv2 = GATConv(hidden_dim, hidden_dim, edge_dim=edge_feature_dim)
        self.lstm = nn.LSTM(input_size=hidden_dim, hidden_size=lstm_hidden_dim, batch_first=True)
        self.fc = nn.Linear(lstm_hidden_dim, output_dim)

    def forward(self, data):
        x, edge_index, edge_attr, trajectory = data.x, data.edge_index, data.edge_attr, data.trajectory
        x = self.conv1(x, edge_index, edge_attr)
        x = F.relu(x)
        x = self.conv2(x, edge_index, edge_attr)
        x = F.relu(x)
        x = F.dropout(x, p=0.2)
        traj_embeddings = x[trajectory]
        lstm_out, (hn, cn) = self.lstm(traj_embeddings.unsqueeze(0))
        lstm_out = lstm_out.squeeze(0)
        out = self.fc(lstm_out[-1])
        return out


In [68]:
from torch_geometric.loader import DataLoader

# Split dataset into train and test (e.g., 80-20 split)
train_size = int(0.8 * len(dataset))
train_dataset = dataset[:train_size]
test_dataset = dataset[train_size:]

# Create DataLoaders
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

In [71]:
# Initialize model, loss, and optimizer
node_feature_dim = dataset[0].x.shape[1]  # Number of node features
edge_feature_dim = dataset[1].x.shape[1]
hidden_dim = 64
lstm_hidden_dim = 32
output_dim = 1  # Example for regression
model = GNN_LSTM(node_feature_dim, edge_feature_dim=0, hidden_dim=hidden_dim, 
                 lstm_hidden_dim=lstm_hidden_dim, output_dim=output_dim)

optimizer = torch.optim.Adam(model.parameters(), lr=0.01)
criterion = nn.MSELoss()  # For regression; use CrossEntropyLoss for classification

# Training loop
def train():
    model.train()
    total_loss = 0
    for data in train_loader:
        optimizer.zero_grad()
        out = model(data)
        loss = criterion(out, data.y)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    return total_loss / len(train_loader)

# Evaluation loop
def test():
    model.eval()
    total_loss = 0
    with torch.no_grad():
        for data in test_loader:
            out = model(data)
            loss = criterion(out, data.y)
            total_loss += loss.item()
    return total_loss / len(test_loader)

# Early stopping parameters
patience = 5  # Number of epochs to wait for improvement
best_test_loss = float('inf')
patience_counter = 0
best_model_path = "best_gnn_lstm_model.pth"

# Train for epochs with early stopping
epochs = 50
for epoch in range(epochs):
    train_loss = train()
    test_loss = test()
    print(f"Epoch {epoch+1}, Train Loss: {train_loss:.4f}, Test Loss: {test_loss:.4f}")
    
    # Check for early stopping
    if test_loss < best_test_loss:
        best_test_loss = test_loss
        patience_counter = 0
        # Save the best model
        torch.save(model.state_dict(), best_model_path)
    else:
        patience_counter += 1
        if patience_counter >= patience:
            print(f"Early stopping triggered after {epoch+1} epochs.")
            break

# Load the best model
model.load_state_dict(torch.load(best_model_path))
print("Loaded best model with test loss: {:.4f}".format(best_test_loss))

c:\Users\brake\UVA_AML24\week_1\.conda\Lib\site-packages\torch\nn\modules\loss.py:610: UserWarning: Using a target size (torch.Size([32])) that is different to the input size (torch.Size([1])). This will likely lead to incorrect results due to broadcasting. Please ensure they have the same size.
  return F.mse_loss(input, target, reduction=self.reduction)
c:\Users\brake\UVA_AML24\week_1\.conda\Lib\site-packages\torch\nn\modules\loss.py:610: UserWarning: Using a target size (torch.Size([26])) that is different to the input size (torch.Size([1])). This will likely lead to incorrect results due to broadcasting. Please ensure they have the same size.
  return F.mse_loss(input, target, reduction=self.reduction)
c:\Users\brake\UVA_AML24\week_1\.conda\Lib\site-packages\torch\nn\modules\loss.py:610: UserWarning: Using a target size (torch.Size([23])) that is different to the input size (torch.Size([1])). This will likely lead to incorrect results due to broadcasting. Please ensure they have th

Epoch 1, Train Loss: 0.1259, Test Loss: 0.1023
Epoch 2, Train Loss: 0.1257, Test Loss: 0.1056
Epoch 3, Train Loss: 0.1258, Test Loss: 0.1024
Epoch 4, Train Loss: 0.1258, Test Loss: 0.1035
Epoch 5, Train Loss: 0.1258, Test Loss: 0.1049
Epoch 6, Train Loss: 0.1258, Test Loss: 0.1033
Early stopping triggered after 6 epochs.
Loaded best model with test loss: 0.1023
